# Schema canónico CAPTIA en InfluxDB — measurement, tags, field, buckets

> _Tutorial · Caso de uso: **Overview** · Capa Medallion: **plata** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Aprender a leer y construir el schema canónico CAPTIA: measurement `captia_point`, 5 tags indexados, field `value` y los 7 buckets con sus retenciones. Trabajar con line protocol y validar la cardinalidad.


## 2. Qué se aprende

- Estructura de InfluxDB 2.7 (measurement, tag, field).
- Por qué CAPTIA usa **un solo measurement** en lugar de uno por variable.
- Cómo se mapea un payload MQTT a una línea de line protocol.
- Qué bucket destino corresponde a cada `metric_kind`.
- Cómo validar el schema con `validate_canonical_tags`.


## 3. Contexto del caso de uso

Toda la telemetría continua de CENTINELA+ vive en `captia_point`. La variable que se mide se identifica por el tag `variable`, no por el nombre del field. Esta decisión mantiene la cardinalidad baja y permite añadir variables nuevas sin tocar el schema.


## 4. Relación con CENTINELA+

El gateway BMS de AULA01 publica cada 5 segundos en topics como `captia/prod/default/ies_simarro/AULA01/telemetry/co2`. Telegraf parsea el topic con una regex de 5 grupos y emite a InfluxDB la línea correspondiente.


## 5. Relación con Medallion

Este notebook es la **especificación operacional de la capa plata**. Toda transformación bronce → plata produce líneas como las de aquí.


## 6. Datos de entrada

Construiremos manualmente 3 puntos sintéticos para AULA01 y los imprimiremos en line protocol.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

Importamos las constantes del repo y mostramos los 7 buckets.


In [2]:
import json
print("MEASUREMENT:", MEASUREMENT_TELEMETRY)
print("CANONICAL TAGS:", CANONICAL_TAGS)
print("FAULT LABELS MEASUREMENT:", MEASUREMENT_FAULT_LABELS)
print(json.dumps(DEFAULT_BUCKET_RETENTIONS, indent=2))


MEASUREMENT: captia_point
CANONICAL TAGS: ('captia_env', 'domain_id', 'site_id', 'asset_id', 'variable')
FAULT LABELS MEASUREMENT: captia_fault_labels
{
  "telemetry": "14d",
  "telemetry_1m": "30d",
  "telemetry_15m": "90d",
  "telemetry_1h": "365d",
  "state_events": "90d",
  "telemetry_events": "90d",
  "captia_metadata": "infinite"
}


## 9. Carga de datos o mock

Generamos 3 puntos: CO₂ a 712 ppm, T_indoor a 22.4 °C y un `ac_state` que se enrutará a `state_events`.


In [3]:
puntos_mock = [
    {
        "topic": build_topic(env="dev", tenant="default", site="ies_simarro",
                              asset="AULA01", variable="co2"),
        "ts_ns": int(pd.Timestamp("2026-05-10T08:00:00Z").value),
        "value": 712.0,
        "metric_kind": "analog_gauge",
    },
    {
        "topic": build_topic(env="dev", tenant="default", site="ies_simarro",
                              asset="AULA01", variable="temperature_01"),
        "ts_ns": int(pd.Timestamp("2026-05-10T08:00:00Z").value),
        "value": 22.4,
        "metric_kind": "analog_gauge",
    },
    {
        "topic": build_topic(env="dev", tenant="default", site="ies_simarro",
                              asset="AULA01", variable="ac_state"),
        "ts_ns": int(pd.Timestamp("2026-05-10T08:00:00Z").value),
        "value": 1.0,
        "metric_kind": "bool_state",
    },
]
puntos_mock


[{'topic': 'captia/dev/default/ies_simarro/AULA01/telemetry/co2',
  'ts_ns': 1778400000000000000,
  'value': 712.0,
  'metric_kind': 'analog_gauge'},
 {'topic': 'captia/dev/default/ies_simarro/AULA01/telemetry/temperature_01',
  'ts_ns': 1778400000000000000,
  'value': 22.4,
  'metric_kind': 'analog_gauge'},
 {'topic': 'captia/dev/default/ies_simarro/AULA01/telemetry/ac_state',
  'ts_ns': 1778400000000000000,
  'value': 1.0,
  'metric_kind': 'bool_state'}]

## 10. Exploración paso a paso

Convertimos cada punto en line protocol y validamos los tags.


In [4]:
def to_line(p):
    parts = p["topic"].split("/")
    tags = {
        "captia_env": parts[1], "domain_id": "bms_classrooms",
        "site_id": parts[3], "asset_id": parts[4], "variable": parts[6],
    }
    validate_canonical_tags(tags)
    return build_line_protocol(
        measurement=MEASUREMENT_TELEMETRY, tags=tags,
        fields={"value": float(p["value"])}, timestamp_ns=p["ts_ns"],
    )

for p in puntos_mock:
    print(to_line(p))


captia_point,asset_id=AULA01,captia_env=dev,domain_id=bms_classrooms,site_id=ies_simarro,variable=co2 value=712.0 1778400000000000000
captia_point,asset_id=AULA01,captia_env=dev,domain_id=bms_classrooms,site_id=ies_simarro,variable=temperature_01 value=22.4 1778400000000000000
captia_point,asset_id=AULA01,captia_env=dev,domain_id=bms_classrooms,site_id=ies_simarro,variable=ac_state value=1.0 1778400000000000000


## 11. Transformación bronce → plata

Las señales `_state` viajan por el mismo topic pero Telegraf las duplica al bucket `state_events` con dedup (solo en cambios de valor). Mostramos cómo se decide.


In [5]:
def bucket_destino(metric_kind: str) -> str:
    if metric_kind in {"bool_state", "setpoint_step"}:
        return "state_events"
    return "telemetry"

for p in puntos_mock:
    print(p["topic"].split("/")[-1], "->", bucket_destino(p["metric_kind"]))


co2 -> telemetry
temperature_01 -> telemetry
ac_state -> state_events


## 12. Construcción de capa oro

No aplica para este notebook (focalizado en la capa plata).


## 13. Visualizaciones explicativas

Pintamos un diagrama de cardinalidad para mostrar cómo crecen las series cuando añadimos un nuevo `asset_id`.


In [6]:
def n_series(asset_count: int, variables: int = 22) -> int:
    return asset_count * variables  # 5 tags fijos, solo asset_id y variable varían

asset_grid = list(range(1, 21))
series = [n_series(a) for a in asset_grid]
plt.figure(figsize=(7, 3))
plt.plot(asset_grid, series, marker="o", color="#3F51B5")
plt.title("Cardinalidad: asset_id × variable")
plt.xlabel("número de aulas")
plt.ylabel("series únicas")
plt.tight_layout()


## 14. Validaciones

Confirmamos que el schema construido cumple las invariantes del repo.


In [7]:
assert MEASUREMENT_TELEMETRY == "captia_point"
assert set(CANONICAL_TAGS) == {"captia_env", "domain_id", "site_id", "asset_id", "variable"}
linea = to_line(puntos_mock[0])
assert linea.startswith("captia_point,")
assert "captia_env=dev" in linea
assert "value=712.0" in linea
print("Schema OK:", linea)


Schema OK: captia_point,asset_id=AULA01,captia_env=dev,domain_id=bms_classrooms,site_id=ies_simarro,variable=co2 value=712.0 1778400000000000000


## 15. Errores comunes

1. **Tag faltante**: si Telegraf falla al parsear el topic, ese punto se descarta silenciosamente. Verificar con `select(stat=count)` por aula.
2. **Tipos inconsistentes**: si una variable se publica a veces como int y a veces como float, InfluxDB rechazará puntos.
3. **Cardinalidad explosiva**: añadir un tag `room_color` con 200 valores diferentes infla la TSDB.
4. **Topic incorrecto**: omitir `/telemetry/` en el medio invalida el routing.


## 16. Ejercicios propuestos

1. Construye line protocol para una variable nueva `air_change_per_hour`.
2. Modifica `validate_canonical_tags` para hacer la advertencia de tags extras un error duro.
3. Escribe una función `parse_topic(topic)` que extraiga los 5 tags.


## 17. Cómo se reutiliza con datos reales

Para `simarro-prod` los tags son idénticos; solo cambia `captia_env=prod`. Las queries Flux que escribiremos en notebooks posteriores funcionan tal cual.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `00_project_overview/02_conexion_influxdb_y_variables_entorno.ipynb`.
- Documento web del caso: `docs/contracts/influx-schema.md`.


## 19. Marco teórico (nivel doctoral)

### Arquitectura Medallion (Databricks 2021)

Tres capas con contratos de calidad incrementales:

$$
\mathcal{L}_b \xrightarrow{\text{ETL}_{b\to s}} \mathcal{L}_s \xrightarrow{\text{ETL}_{s\to o}} \mathcal{L}_o
$$

donde
$$
\mathcal{Q}(\mathcal{L}_o) > \mathcal{Q}(\mathcal{L}_s) > \mathcal{Q}(\mathcal{L}_b)
$$

con $\mathcal{Q}$ score de calidad ($\in [0, 1]$) según reglas del Caso G.

### Schema canónico CAPTIA

Modelo dimensional de **un único** measurement con 5 tags:

$$
\text{captia\_point}(\underbrace{e, d, s, a, v}_{5\ \text{tags}}, \underbrace{t}_{\text{ts\_ns}}, \underbrace{x}_{\text{value}})
$$

con cardinalidad efectiva
$$
|\mathcal{S}| = |E| \cdot |D| \cdot |S| \cdot |A| \cdot |V| \approx 3 \times 5 \times 10 \times 70 \times 24 = 252\,000
$$
series únicas como cota superior — en práctica decenas de miles.

### Reproducibilidad determinista

$$
\hat{y} = M(D, \theta, s = 42) \implies \text{hash}_2(\hat{y}_1) = \text{hash}_2(\hat{y}_2)
$$

con $s$ semilla, $D$ dataset (versionado lakeFS), $\theta$ hiperparámetros.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

**CAPTIA Technology** mantiene un schema canónico único en CENTINELA+. Este repo extiende ese schema a un **generador sintético reproducible** que permite:

- Entrenar modelos de los casos B/C/D antes de tener histórico real del IES Simarro.
- Onboarding de centros nuevos sin esperar meses de captura.
- Datasets etiquetados de fallos HVAC para Caso C, normalmente imposibles de obtener.
- Material docente alineado 1:1 con producción.

### ROI estimado

| Beneficio | Valor anual |
|---|---|
| Onboarding de 5 centros sin coste de captura | +25 000 € |
| Material docente reutilizable curso a curso | +8 000 € |
| Aceleración POC IA al cliente final | +12 000 € |
| **Beneficio bruto** | **+45 000 €/año** |
| Coste mantenimiento del repo | -3 000 €/año |
| **Neto** | **+42 000 €/año** |

### Riesgos y mitigaciones

- Calibración inicial sin datos reales del IES Simarro (mitigado con `docs/specs/digital-twin-bms-physics-validation/`).
- Drift entre generador y producción: validado por suite 211/211 PASS y score realismo físico 0.94.


## 21. Bibliografía y referencias

- Inmon, W. H., Linstedt, D., & Levins, M. (2019). *Data Architecture: A Primer for the Data Scientist*. Academic Press.
- Databricks (2021). *Lakehouse: A New Generation of Open Platforms*. CIDR.
- InfluxData (2024). *InfluxDB 2.7 Reference Architecture*. https://docs.influxdata.com/influxdb/v2/
- ECMWF (2024). *ERA5 reanalysis dataset documentation*. Copernicus Climate Change Service.
